<a href="https://colab.research.google.com/github/koenga/Projet-CE-1/blob/main/DLAV_Phase2_%C3%A0_tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Trajectory Prediction with Auxiliary Depth Estimation

# 🧭 Introduction

"""
Welcome to **Phase 2** of the DLAV Projec! 🚗💨

In this phase, you'll work with a more challenging dataset that includes:
- RGB **camera images**
- Ground-truth **depth maps**
- Ground-truth **semantic segmentation** labels

Your goal is still to predict the **future trajectory** of the self-driving car (SDC), but you now have more tools at your disposal! 🎯

Here, we provide an example where **depth estimation** is used as an auxiliary task to improve trajectory prediction.

However, you're **free to explore** other auxiliary tasks (e.g., using semantic labels), different loss functions, data augmentations, or better architectures! 💡

This notebook will walk you through loading the dataset, building a model, training with and without the auxiliary task, and visualizing results.
"""

In [1]:
# Install gdown to handle Google Drive file download
!pip install -q gdown

import gdown
import zipfile

download_url = f"https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr"
output_zip = "dlav_train.zip"
gdown.download(download_url, output_zip, quiet=False)  # Downloads the file to your drive
with zipfile.ZipFile(output_zip, 'r') as zip_ref:  # Extracts the downloaded zip file
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu"
output_zip = "dlav_val.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(".")

download_url = "https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV"
output_zip = "dlav_test_public.zip"
gdown.download(download_url, output_zip, quiet=False)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(".")

Downloading...
From (original): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr
From (redirected): https://drive.google.com/uc?id=1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr&confirm=t&uuid=bb7628ec-362c-4d0f-adf1-679288c787db
To: /content/dlav_train.zip
100%|██████████| 439M/439M [00:06<00:00, 69.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu
From (redirected): https://drive.google.com/uc?id=1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu&confirm=t&uuid=0d01948c-160e-4b19-ab4b-2dc444aff179
To: /content/dlav_val.zip
100%|██████████| 87.8M/87.8M [00:01<00:00, 52.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV
From (redirected): https://drive.google.com/uc?id=1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV&confirm=t&uuid=2dc79ebe-9e04-457b-adf0-39b212228626
To: /content/dlav_test_public.zip
100%|██████████| 86.6M/86.6M [00:01<00:00, 49.4MB/s]


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📂 The Dataset

We are now working with a richer dataset that includes not just images and trajectories,
but also **depth maps** (and semantic segmentation labels, though unused in this example).

The data is stored in `.pkl` files and each file contains:
- `camera`: RGB image (shape: H x W x 3)
- `sdc_history_feature`: the past trajectory of the car
- `sdc_future_feature`: the future trajectory to predict
- `depth`: ground truth depth map (shape: H x W x 1)

We'll define a `DrivingDataset` class to load and return these tensors in a format our model can work with.

In [12]:
import os
import torch
import pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import csv
import random
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

class DrivingDataset(Dataset):
    def __init__(self, file_list, test=False):
        self.samples = file_list
        self.test = test

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        with open(self.samples[idx], 'rb') as f:
            data = pickle.load(f)

        camera = torch.FloatTensor(data['camera']).permute(2, 0, 1) / 255.0
        history = torch.FloatTensor(data['sdc_history_feature'])

        depth = torch.FloatTensor(data['depth']).permute(2, 0, 1)

        # Coordonnées relatives (x, y)
        last_pos = history[-1, :2].clone()
        last_heading = history[-1, 2].clone()

        history[:, :2] -= last_pos
        history[:, 2]  -= last_heading

        item = {
            'camera':  camera,
            'history': history,
            'depth':   depth,
        }

        if 'semantic_label' in data:
            # (H, W) — entiers de 0 à 14
            item['semantic_label'] = torch.LongTensor(data['semantic_label'])

        if not self.test:
            future = torch.FloatTensor(data['sdc_future_feature'])
            future[:, :2] -= last_pos
            future[:, 2]  -= last_heading

            item['future']   = future
            item['last_pos'] = last_pos
            item['last_heading'] = last_heading

        return item

## 🧠 The Model: Trajectory + Depth Prediction

We've extended our trajectory prediction model to optionally include a **depth estimation decoder**.

Why?
- Predicting depth helps the model **learn richer visual features** from the camera input.
- This acts as a form of **multi-task learning**, where learning to estimate depth reinforces scene understanding, ultimately leading to better trajectory predictions.
- This can be especially useful in complex environments with occlusions or sharp turns.

The model has:
- A CNN backbone to extract features from the image
- An MLP to process historical trajectory features
- A trajectory decoder to predict future coordinates
- (Optionally) A depth decoder to predict dense depth maps

This auxiliary task is enabled by setting `use_depth_aux=True`.

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class CrossAttentionFusion(nn.Module):
    """Un bloc de cross-attention : historique (query) × carte spatiale (key/value)."""
    def __init__(self, feat_channels, hist_dim=256, num_heads=8):
        super().__init__()
        self.proj = nn.Conv2d(feat_channels, hist_dim, kernel_size=1)
        self.attn = nn.MultiheadAttention(embed_dim=hist_dim, num_heads=num_heads,
                                           batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(hist_dim)

    def forward(self, feat_map, x_hist):
        feat = self.proj(feat_map)                      # (B, hist_dim, H, W)
        seq  = feat.flatten(2).transpose(1, 2)          # (B, H*W, hist_dim)
        q    = x_hist.unsqueeze(1)                      # (B, 1, hist_dim)
        out, _ = self.attn(q, seq, seq)                 # (B, 1, hist_dim)
        return self.norm(out.squeeze(1) + x_hist)       # résiduel (B, hist_dim)

    def forward(self, feat_map, x_hist):
        """
        feat_map : (B, C, H, W)
        x_hist   : (B, hist_dim)
        retourne : (B, hist_dim)
        """
        feat = self.proj(feat_map)                          # (B, hist_dim, H, W)
        seq  = feat.flatten(2).transpose(1, 2)              # (B, H*W, hist_dim)
        q    = x_hist.unsqueeze(1)                          # (B, 1, hist_dim)
        out, _ = self.attn(q, seq, seq)                     # (B, 1, hist_dim)
        return self.norm(out.squeeze(1) + x_hist)           # résiduel (B, hist_dim)

class DrivingPlanner(nn.Module):
    def __init__(self, use_depth_aux=False, use_seg_aux=False):
        super().__init__()
        self.use_depth_aux = use_depth_aux
        self.use_seg_aux   = use_seg_aux

        HIST_DIM = 256  # augmenté de 128 → 256

        # Backbone ResNet34
        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # (B,  64, H/4,  W/4)
        self.layer2 = backbone.layer2   # (B, 128, H/8,  W/8)
        self.layer3 = backbone.layer3   # (B, 256, H/16, W/16)
        self.layer4 = backbone.layer4   # (B, 512, H/32, W/32)

        # GRU historique — hidden_size augmenté à 256
        self.hist_gru = nn.GRU(input_size=3, hidden_size=HIST_DIM,
                                num_layers=2, batch_first=True, dropout=0.1)

        # Blocs de cross-attention à 3 niveaux (num_heads=8 car hist_dim=256)
        self.fuse2 = CrossAttentionFusion(feat_channels=128, hist_dim=HIST_DIM, num_heads=8)
        self.fuse3 = CrossAttentionFusion(feat_channels=256, hist_dim=HIST_DIM, num_heads=8)
        self.fuse4 = CrossAttentionFusion(feat_channels=512, hist_dim=HIST_DIM, num_heads=8)

        # Fusion : 3 contextes (256 chacun) + x_hist (256) = 1024
        fusion_dim = HIST_DIM * 3 + HIST_DIM  # 1024

        # Decoder plus large, dropout réduit
        self.decoder = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 60 * 3),
        )

        # Tête auxiliaire : depth
        if self.use_depth_aux:
            self.depth_proj = nn.Conv2d(512, 32, kernel_size=1)
            self.depth_decoder = nn.Sequential(
                nn.ConvTranspose2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
                nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
                nn.Conv2d(16,  1, kernel_size=3, padding=1),
                nn.Upsample(size=(200, 300), mode='bilinear', align_corners=False),
            )

        # Tête auxiliaire : segmentation
        if self.use_seg_aux:
            self.seg_proj = nn.Conv2d(512, 64, kernel_size=1)
            self.seg_decoder = nn.Sequential(
                nn.ConvTranspose2d( 64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),
                nn.ConvTranspose2d(128,  64, kernel_size=4, stride=2, padding=1), nn.ReLU(),
                nn.Conv2d(64, 15, kernel_size=3, padding=1),
                nn.Upsample(size=(200, 300), mode='bilinear', align_corners=False),
            )

    def forward(self, camera, history):
        B = camera.size(0)

        # Normalisation ImageNet
        mean = torch.tensor([0.485, 0.456, 0.406], device=camera.device).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=camera.device).view(1, 3, 1, 1)
        camera = (camera - mean) / std

        # Features multi-niveaux
        x     = self.stem(camera)
        feat1 = self.layer1(x)
        feat2 = self.layer2(feat1)
        feat3 = self.layer3(feat2)
        feat4 = self.layer4(feat3)

        # Têtes auxiliaires
        depth_out = None
        if self.use_depth_aux:
            depth_out = self.depth_decoder(self.depth_proj(feat4))

        seg_out = None
        if self.use_seg_aux:
            seg_out = self.seg_decoder(self.seg_proj(feat4))

        # Historique
        _, h   = self.hist_gru(history)
        x_hist = h[-1]  # (B, 256)

        # Fusion multi-niveaux
        ctx2 = self.fuse2(feat2, x_hist)  # (B, 256)
        ctx3 = self.fuse3(feat3, x_hist)  # (B, 256)
        ctx4 = self.fuse4(feat4, x_hist)  # (B, 256)

        x      = torch.cat([ctx2, ctx3, ctx4, x_hist], dim=1)  # (B, 1024)
        future = self.decoder(x).view(B, 60, 3)

        return future, depth_out, seg_out


## 🏋️ Training with Auxiliary Loss

The training loop is similar to Phase 1 — except now, if enabled, we also compute a loss on the predicted **depth map**.

We define:
- `trajectory_loss` as standard MSE between predicted and true future trajectory
- `depth_loss` as L1 loss between predicted and ground truth depth

Total loss = `trajectory_loss + lambda * depth_loss`

This helps guide the model to learn better representations from visual input. The weight `lambda` is a hyperparameter you can tune!

In [14]:
import torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR

def warmup_lr(optimizer, epoch, warmup_epochs, base_lrs):
    """Warmup du learning rate"""
    scale = (epoch + 1) / warmup_epochs
    for pg, base_lr in zip(optimizer.param_groups, base_lrs):
        pg['lr'] = base_lr * scale

def train_one_epoch(model, train_loader, optimizer, device, lambda_depth=0.05, lambda_seg=0.1, use_depth_aux=False, use_seg_aux=False, epoch=0, warmup_epochs=3, base_lrs=None):
    model.train()
    train_loss = traj_loss_sum = depth_loss_sum = seg_loss_sum = 0.0

    # Warmup manuel
    if epoch < warmup_epochs and base_lrs is not None:
        warmup_lr(optimizer, epoch, warmup_epochs, base_lrs)

    for batch in train_loader:
        cam  = batch['camera'].to(device)
        hist = batch['history'].to(device)
        fut  = batch['future'].to(device)

        optimizer.zero_grad()

        future_pred, depth_pred, seg_pred = model(cam, hist)

        # Loss trajectoire (XY seulement)
        traj_loss = F.mse_loss(future_pred[..., :2], fut[..., :2])
        loss = traj_loss
        traj_loss_sum += traj_loss.item()

        # Loss depth (optionnelle)
        if use_depth_aux and depth_pred is not None and 'depth' in batch:
            dep = batch['depth'].to(device)
            depth_loss = F.l1_loss(depth_pred, dep)
            loss = loss + lambda_depth * depth_loss
            depth_loss_sum += depth_loss.item()

        # Loss segmentation (optionnelle)
        if use_seg_aux and seg_pred is not None and 'semantic_label' in batch:
            seg = batch['semantic_label'].to(device).long()  # (B, H, W)
            seg_loss = F.cross_entropy(seg_pred, seg)
            loss = loss + lambda_seg * seg_loss
            seg_loss_sum += seg_loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    n = len(train_loader)
    return (
        train_loss / n,
        traj_loss_sum / n,
        depth_loss_sum / n if use_depth_aux else None,
        seg_loss_sum / n if use_seg_aux else None,
    )

def validate(model, val_loader, device, use_depth_aux=False, use_seg_aux=False):
    model.eval()
    total_ade = total_fde = total_mse = 0.0
    total_depth_loss = 0.0
    total_seg_loss   = 0.0
    count = 0

    with torch.no_grad():
        for batch in val_loader:
            cam  = batch['camera'].to(device)
            hist = batch['history'].to(device)
            fut  = batch['future'].to(device)

            future_pred, depth_pred, seg_pred = model(cam, hist)

            B, T, _ = fut.shape
            count += B

            # Métriques trajectoire (XY)
            ade = torch.norm(future_pred[:, :, :2] - fut[:, :, :2], dim=2).mean(dim=1).sum()
            fde = torch.norm(future_pred[:, -1, :2] - fut[:, -1, :2], dim=1).sum()
            mse = F.mse_loss(future_pred[..., :2], fut[..., :2], reduction='sum')

            total_ade += ade.item()
            total_fde += fde.item()
            total_mse += mse.item()

            # Métriques auxiliaires (si dispo)
            if use_depth_aux and depth_pred is not None and 'depth' in batch:
                dep = batch['depth'].to(device)
                total_depth_loss += F.l1_loss(depth_pred, dep).item()

            if use_seg_aux and seg_pred is not None and 'semantic_label' in batch:
                seg = batch['semantic_label'].to(device).long()
                total_seg_loss += F.cross_entropy(seg_pred, seg).item()

    n_batches = len(val_loader)
    ade_avg   = total_ade / count
    fde_avg   = total_fde / count
    mse_avg   = total_mse / (count * 60 * 2)

    return (ade_avg, fde_avg, mse_avg, total_depth_loss / n_batches if use_depth_aux else None, total_seg_loss   / n_batches if use_seg_aux   else None,)

def train(model, train_loader, val_loader, optimizer, num_epochs=50,
          lambda_depth=0.05, lambda_seg=0.1, use_depth_aux=False,
          use_seg_aux=False, warmup_epochs=3, experiment_name="experiment"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = model.to(device)

    # LR de base pour le warmup (une valeur par groupe de paramètres)
    base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    # Scheduler cosine (démarre après le warmup)
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=max(num_epochs - warmup_epochs, 1),
        eta_min=1e-5,
    )

    history = {
        'train_loss':       [],
        'train_traj_loss':  [],
        'train_depth_loss': [] if use_depth_aux else None,
        'train_seg_loss':   [] if use_seg_aux   else None,
        'val_ade':          [],
        'val_fde':          [],
        'val_mse':          [],
        'val_depth_loss':   [] if use_depth_aux else None,
        'val_seg_loss':     [] if use_seg_aux   else None,
        'best_ade':         float('inf'),
    }

    best_model_path = (
        f"/content/drive/MyDrive/Colab Notebooks/Project Phase 2/"
        f"best_model_{experiment_name}.pth"
    )

    for epoch in range(num_epochs):
        # ----- Training -----
        train_loss, traj_loss, depth_loss, seg_loss = train_one_epoch(
            model, train_loader, optimizer, device,
            lambda_depth=lambda_depth, lambda_seg=lambda_seg,
            use_depth_aux=use_depth_aux, use_seg_aux=use_seg_aux,
            epoch=epoch, warmup_epochs=warmup_epochs, base_lrs=base_lrs,
        )
        history['train_loss'].append(train_loss)
        history['train_traj_loss'].append(traj_loss)
        if use_depth_aux:
            history['train_depth_loss'].append(depth_loss)
        if use_seg_aux:
            history['train_seg_loss'].append(seg_loss)

        # Cosine scheduler (uniquement après le warmup)
        if epoch >= warmup_epochs:
            scheduler.step()

        # ----- Validation -----
        ade, fde, mse, depth_val, seg_val = validate(
            model, val_loader, device,
            use_depth_aux=use_depth_aux,
            use_seg_aux=use_seg_aux,
        )
        history['val_ade'].append(ade)
        history['val_fde'].append(fde)
        history['val_mse'].append(mse)
        if use_depth_aux:
            history['val_depth_loss'].append(depth_val)
        if use_seg_aux:
            history['val_seg_loss'].append(seg_val)

        # ----- Logging -----
        current_lr = optimizer.param_groups[-1]['lr']
        aux_str = ""
        if use_depth_aux and depth_loss is not None:
            aux_str += f", Depth={depth_loss:.4f}"
        if use_seg_aux and seg_loss is not None:
            aux_str += f", Seg={seg_loss:.4f}"

        val_aux_str = ""
        if use_depth_aux and depth_val is not None:
            val_aux_str += f", DepthVal={depth_val:.4f}"
        if use_seg_aux and seg_val is not None:
            val_aux_str += f", SegVal={seg_val:.4f}"

        print(
            f"Epoch [{epoch+1:3d}/{num_epochs}] lr={current_lr:.2e} | "
            f"Loss: {train_loss:.4f} (Traj: {traj_loss:.4f}{aux_str}) | "
            f"Val: ADE={ade:.3f}, FDE={fde:.3f}, MSE={mse:.6f}{val_aux_str}"
        )

        # ----- Sauvegarde du meilleur modèle -----
        if ade < history['best_ade']:
            history['best_ade'] = ade
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'ade':  ade,
                'fde':  fde,
                'mse':  mse,
                'history': history,
            }, best_model_path)
            print(f"  -> ✓ Best model saved (ADE: {ade:.3f})")

    print(f"\n{'='*60}")
    print(f"Best ADE: {history['best_ade']:.3f}")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*60}\n")
    return model, history


In [15]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim import AdamW
import os

def build_loaders(train_dir="train", val_dir="val", batch_size=32):
    train_files = [os.path.join(train_dir, f)
                   for f in os.listdir(train_dir) if f.endswith('.pkl')]
    val_files   = [os.path.join(val_dir, f)
                   for f in os.listdir(val_dir)   if f.endswith('.pkl')]

    train_loader = DataLoader(
        DrivingDataset(train_files),
        batch_size=batch_size, shuffle=True, num_workers=2,
    )
    val_loader = DataLoader(
        DrivingDataset(val_files),
        batch_size=batch_size, shuffle=False, num_workers=2,
    )
    return train_loader, val_loader

def build_optimizer(model, backbone_lr=1e-4, head_lr=1e-3, weight_decay=1e-4):
    backbone_params = []
    for attr in ('stem', 'layer1', 'layer2', 'layer3', 'layer4'):
        backbone_params += list(getattr(model, attr).parameters())

    head_params = []
    for attr in ('hist_gru', 'fuse2', 'fuse3', 'fuse4', 'decoder'):
        head_params += list(getattr(model, attr).parameters())
    for attr in ('depth_proj', 'depth_decoder', 'seg_proj', 'seg_decoder'):
        if hasattr(model, attr):
            head_params += list(getattr(model, attr).parameters())

    return optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr},
        {'params': head_params,     'lr': head_lr},
    ], weight_decay=weight_decay)

## 🔍 Let's Compare Two Settings

We'll now train and evaluate the model in **two modes**:

1. **Without auxiliary task** — the model only predicts the trajectory.
2. **With depth auxiliary task** — the model also predicts a depth map, which helps it learn better visual features.

By comparing the results (ADE, FDE, and Trajectory MSE), you'll see the benefit of multi-task learning in action! 🚀

In [ ]:
if __name__ == "__main__":
    train_loader, val_loader = build_loaders()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    NUM_EPOCHS    = 150
    WARMUP_EPOCHS = 5

# ── Expérience 1 : baseline GRU, sans auxiliaires ──────────────────────
    # print("\n" + "="*60)
    # print("Expérience 1 : baseline GRU (sans auxiliaires)")
    # print("="*60)

    # model_base = DrivingPlanner(use_depth_aux=False, use_seg_aux=False)
    # opt_base   = build_optimizer(model_base)

    # model_base, hist_base = train(
    #     model_base, train_loader, val_loader, opt_base,
    #     num_epochs=NUM_EPOCHS,
    #     use_depth_aux=False, use_seg_aux=False,
    #     warmup_epochs=WARMUP_EPOCHS,
    #     experiment_name="gru_no_aux",
    # )

# ── Expérience 2 : GRU + depth + segmentation ──────────────────────────
    print("\n" + "="*60)
    print("Expérience 2 : GRU + depth + segmentation")
    print("="*60)

    model_full = DrivingPlanner(use_depth_aux=True, use_seg_aux=True)
    opt_full   = build_optimizer(model_full)

    model_full, hist_full = train(
        model_full, train_loader, val_loader, opt_full,
        num_epochs=NUM_EPOCHS,
        lambda_depth=0.01, lambda_seg=0.1,
        use_depth_aux=True, use_seg_aux=True,
        warmup_epochs=WARMUP_EPOCHS,
        experiment_name="transfuser_v2",
    )

# ── Résumé final ────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Résumé des meilleurs ADE de validation :")
    #print(f"  Baseline GRU       : {hist_base['best_ade']:.3f}")
    print(f"  GRU + depth + seg  : {hist_full['best_ade']:.3f}")
    print("="*60)


Expérience 2 : GRU + depth + segmentation
Using device: cuda
Epoch [  1/150] lr=2.00e-04 | Loss: 54.7319 (Traj: 53.4091, Depth=122.4148, Seg=0.9873) | Val: ADE=3.131, FDE=7.842, MSE=13.905607, DepthVal=40.9061, SegVal=0.5069
  -> ✓ Best model saved (ADE: 3.131)
Epoch [  2/150] lr=4.00e-04 | Loss: 13.2225 (Traj: 12.9185, Depth=26.3655, Seg=0.4027) | Val: ADE=3.489, FDE=8.252, MSE=14.164134, DepthVal=16.1027, SegVal=0.3387
Epoch [  3/150] lr=6.00e-04 | Loss: 12.6498 (Traj: 12.4674, Depth=15.0962, Seg=0.3139) | Val: ADE=2.829, FDE=7.129, MSE=10.532131, DepthVal=16.0093, SegVal=0.2908
  -> ✓ Best model saved (ADE: 2.829)
Epoch [  4/150] lr=8.00e-04 | Loss: 9.9565 (Traj: 9.7931, Depth=13.5341, Seg=0.2811) | Val: ADE=3.867, FDE=9.801, MSE=16.391465, DepthVal=13.6030, SegVal=0.2587
Epoch [  5/150] lr=1.00e-03 | Loss: 10.5398 (Traj: 10.3802, Depth=13.4401, Seg=0.2519) | Val: ADE=2.697, FDE=6.759, MSE=9.871663, DepthVal=10.8057, SegVal=0.2394
  -> ✓ Best model saved (ADE: 2.697)
Epoch [  6/150

In [ ]:
# Charger le checkpoint
ckpt_path = "/content/drive/MyDrive/Colab Notebooks/Project Phase 2/best_model_transfuser_v2.pth"
checkpoint = torch.load(ckpt_path, map_location=device)

# Recréer le modèle et charger les poids
model_full = DrivingPlanner(use_depth_aux=True, use_seg_aux=True)
model_full.load_state_dict(checkpoint['model_state_dict'])

# Optimiseur avec LR bas — on repart depuis epoch ~98
opt_full = build_optimizer(model_full, backbone_lr=5e-6, head_lr=5e-5)

# Relancer pour les epochs restantes
model_full, hist_full = train(
    model_full, train_loader, val_loader, opt_full,
    num_epochs=90,          # epochs restantes (120 - 98 = ~22, prenez 30 pour marge)
    lambda_depth=0.01, lambda_seg=0.1,
    use_depth_aux=True, use_seg_aux=True,
    warmup_epochs=0,        # pas de warmup, on repart d'un modèle entraîné
    experiment_name="transfuser_v2_continued",
)

## 🔍 Final Visualization and Comparison

Now that we’ve trained two models — one **with** the depth auxiliary task and one **without** — let’s visualize and compare their predictions.

We’ll show:
1. The **camera image** from selected validation examples
2. The **past trajectory**, **ground-truth future**, and **predicted future** trajectory
3. The **predicted vs. ground-truth depth maps** (only for the model trained with the auxiliary task)

These visualizations help us understand:
- Does the predicted trajectory better match the future when the depth task is included?
- Is the predicted depth map reasonably accurate?

Let’s see the difference! 📈

In [ ]:
import matplotlib.pyplot as plt
import random

random.seed(40)

def visualize_comparison(val_loader, model_base, model_full, device):
    model_base.eval()
    model_full.eval()
    val_batch = next(iter(val_loader))

    camera = val_batch['camera'].to(device)
    history = val_batch['history'].to(device)
    future = val_batch['future'].to(device)
    depth = val_batch['depth'].to(device)

    with torch.no_grad():
        pred_no_aux, _ = model_base(camera, history)
        pred_with_aux, pred_depth = model_full(camera, history)

    camera = camera.cpu().numpy()
    history = history.cpu().numpy()
    future = future.cpu().numpy()
    pred_no_aux = pred_no_aux.cpu().numpy()
    pred_with_aux = pred_with_aux.cpu().numpy()
    depth = depth.cpu().numpy()
    pred_depth = pred_depth.cpu().numpy() if pred_depth is not None else None

    k = 4
    indices = random.choices(np.arange(len(camera)), k=k)

    # Show the input camera images
    fig, ax = plt.subplots(1, k, figsize=(4 * k, 4))
    for i, idx in enumerate(indices):
        ax[i].imshow(camera[idx].transpose(1, 2, 0))
        ax[i].set_title(f"Example {i+1}")
        ax[i].axis("off")
    plt.suptitle("Camera Inputs")
    plt.tight_layout()
    plt.show()

    # Compare predicted trajectories
    fig, ax = plt.subplots(2, k, figsize=(4 * k, 8))
    for i, idx in enumerate(indices):
        # Without aux
        ax[0, i].plot(history[idx, :, 0], history[idx, :, 1], 'o-', label='Past', color='gold', markersize=4, linewidth=1.2)
        ax[0, i].plot(future[idx, :, 0], future[idx, :, 1], 'o-', label='GT Future', color='green', markersize=4, linewidth=1.2)
        ax[0, i].plot(pred_no_aux[idx, :, 0], pred_no_aux[idx, :, 1], 'o-', label='Pred (No Aux)', color='red', markersize=4, linewidth=1.2)
        ax[0, i].set_title("No Depth Aux")
        ax[0, i].axis("equal")

        # With aux
        ax[1, i].plot(history[idx, :, 0], history[idx, :, 1], 'o-', label='Past', color='gold', markersize=4, linewidth=1.2)
        ax[1, i].plot(future[idx, :, 0], future[idx, :, 1], 'o-', label='GT Future', color='green', markersize=4, linewidth=1.2)
        ax[1, i].plot(pred_with_aux[idx, :, 0], pred_with_aux[idx, :, 1], 'o-', label='Pred (With Aux)', color='blue', markersize=4, linewidth=1.2)
        ax[1, i].set_title("With Depth Aux")
        ax[1, i].axis("equal")

    # Show full legend in a new figure
    fig_legend = plt.figure(figsize=(8, 1))
    legend_handles = [
        plt.Line2D([0], [0], color='gold', marker='o', linestyle='-', markersize=5, label='Past'),
        plt.Line2D([0], [0], color='green', marker='o', linestyle='-', markersize=5, label='GT Future'),
        plt.Line2D([0], [0], color='red', marker='o', linestyle='-', markersize=5, label='Pred (No Aux)'),
        plt.Line2D([0], [0], color='blue', marker='o', linestyle='-', markersize=5, label='Pred (With Aux)')
    ]
    fig_legend.legend(handles=legend_handles, loc='center', ncol=4)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    plt.suptitle("Trajectory Prediction: Without vs With Depth Aux Task")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # Show predicted vs GT depth (only for bottom row)
    if pred_depth is not None:
        fig, ax = plt.subplots(2, k, figsize=(4 * k, 6))
        for i, idx in enumerate(indices):
            ax[0, i].imshow(depth[idx, :, :, 0], cmap='viridis')
            ax[0, i].set_title("GT Depth", pad=10)
            ax[0, i].axis("off")
            # increase vertical distance between rows

            ax[1, i].imshow(pred_depth[idx, :, :, 0], cmap='viridis')
            ax[1, i].set_title("Pred Depth", pad=10)
            ax[1, i].axis("off")

        plt.suptitle("Depth Estimation (Only for Model With Aux Task)", y=1.05)
        plt.subplots_adjust(hspace=0.4)
        plt.tight_layout()
        plt.show()


# 🔚 Call at the end after training both models
visualize_comparison(val_loader, model_base, model_full, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

Now we run our model on the test set once, to get the plan of our model and save it for submission. Notice that the ground truth plans are removed for the test set, so you can not calculate the ADE metric on the test set yourself, and need to submit it to the leader board. By running the last cell, you'll be able to see a csv file called submission_phase2.csv by clicking on the folder icon on the left. Download it and submit it to the leaderboard to get your score.

In [ ]:
from torch.utils.data import DataLoader
# ==================== CHARGER LE MEILLEUR MODÈLE ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Créer et charger le modèle
model = DrivingPlanner(use_depth_aux=True)
checkpoint = torch.load('/content/drive/MyDrive/Colab Notebooks/Project Phase 2/best_model_gru_full_aux_continued.pth', map_location=device)

In [ ]:
# Charger le checkpoint
checkpoint = torch.load('/content/drive/MyDrive/Colab Notebooks/Project Phase 2/best_model_gru_full_aux_continued.pth', map_location=device)

# Filtrer les clés incompatibles
model_state_dict = model.state_dict()
filtered_checkpoint = {}

for key, value in checkpoint['model_state_dict'].items():
    if key in model_state_dict and value.shape == model_state_dict[key].shape:
        filtered_checkpoint[key] = value
    else:
        print(f"Skipping: {key} (shape mismatch or not found)")

# Charger les poids filtrés
model.load_state_dict(filtered_checkpoint, strict=False)
print(f"✅ Loaded {len(filtered_checkpoint)}/{len(model_state_dict)} layers")

In [ ]:
# Charger les poids
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print("✅ Model loaded successfully!")

In [ ]:
with open(f"test_public/0.pkl", "rb") as f:
    data = pickle.load(f)
print(data.keys())
# Note the absence of sdc_future_feature

In [ ]:
import pandas as pd
test_data_dir = "test_public"
test_files = [os.path.join(test_data_dir, fn) for fn in sorted([f for f in os.listdir(test_data_dir) if f.endswith(".pkl")], key=lambda fn: int(os.path.splitext(fn)[0]))]
test_dataset = DrivingDataset(test_files, test=True)
test_loader = DataLoader(test_dataset, batch_size=250, num_workers=2)
model_full.eval()
all_plans = []
with torch.no_grad():
    for batch in test_loader:
        camera = batch['camera'].to(device)
        history = batch['history'].to(device)

        pred_future, _, _ = model_full(camera, history)
        all_plans.append(pred_future.cpu().numpy()[..., :2])
all_plans = np.concatenate(all_plans, axis=0)

# Now save the plans as a csv file
pred_xy = all_plans[..., :2]  # shape: (total_samples, T, 2)

# Flatten to (total_samples, T*2)
total_samples, T, D = pred_xy.shape
pred_xy_flat = pred_xy.reshape(total_samples, T * D)

# Build a DataFrame with an ID column
ids = np.arange(total_samples)
df_xy = pd.DataFrame(pred_xy_flat)
df_xy.insert(0, "id", ids)

# Column names: id, x_1, y_1, x_2, y_2, ..., x_T, y_T
new_col_names = ["id"]
for t in range(1, T + 1):
    new_col_names.append(f"x_{t}")
    new_col_names.append(f"y_{t}")
df_xy.columns = new_col_names

# Save to CSV
df_xy.to_csv("/content/drive/MyDrive/Colab Notebooks/Project Phase 2/submission_phase2.csv", index=False)

print(f"Shape of df_xy: {df_xy.shape}")